# Deforestation and Recovery Balance — Managed Forest of Quebec

This notebook extracts annual Sentinel-2 composites over the Abitibi-Temiscamingue administrative region (managed public forest, Quebec), detects year-over-year change (dNBR), and computes the net balance between lost and recovering forest cover per spatial unit. It also derives aboveground carbon density from GEDI L4B and crosses it with the change classification.

**Pipeline:** GEE extraction -> annual NDVI/NBR composites -> change detection -> spatial aggregation -> net balance -> GEDI carbon density -> carbon exposure in change zones.

In [ ]:
import sys
sys.path.append('../src')

import ee
import geemap
import pandas as pd

from gee_utils import init_ee, build_annual_stack
from change_detection import build_change_series, aggregate_by_units

PROJECT = "your-gee-project-id"
init_ee(PROJECT)

## 1. Study area: Abitibi-Temiscamingue administrative region

Source: "Decoupages administratifs" layer from Donnees Quebec / MRNF (province-wide administrative divisions, 1:1,000,000 scale), filtered to the Abitibi-Temiscamingue region. Download it locally first (see README) and place it under `data/`.

In [ ]:
import geopandas as gpd
import geemap
import glob

# The region polygon layer is regio_s.shp ('_s' = surface/polygons,
# '_l' = boundary lines only) inside the extracted BDGA_1M archive.
shp_candidates = glob.glob("../data/decoupages/**/regio_s.shp", recursive=True)
print("Found:", shp_candidates)
SHP_PATH = shp_candidates[0]

In [ ]:
gdf = gpd.read_file(SHP_PATH)
print("Columns:", list(gdf.columns))
gdf.head()

In [ ]:
# Step 3: identify the field holding the region name
# (check unique values to confirm the exact column and value)
REGION_NAME_FIELD = "NOM"  # adjust based on the columns printed above

print(gdf[REGION_NAME_FIELD].unique())

In [ ]:
# Step 4: filter the region and convert it to a GEE AOI
gdf_region = gdf[gdf[REGION_NAME_FIELD].str.contains("Abitibi", case=False, na=False)]
gdf_region = gdf_region.to_crs("EPSG:4326")

aoi_fc = geemap.geopandas_to_ee(gdf_region)
AOI = aoi_fc.geometry()

Map = geemap.Map()
Map.centerObject(AOI, 8)
Map.addLayer(AOI, {}, "AOI - Abitibi-Temiscamingue")
Map

## 2. Annual composites (NDVI/NBR)

In [ ]:
YEARS = range(2017, 2026)
annual_stack = build_annual_stack(YEARS, AOI)
print(f"Composites generated: {annual_stack.size().getInfo()}")

In [ ]:
# Quick preview of the latest composite (NBR)
last_img = ee.Image(annual_stack.sort('year', False).first())
nbr_vis = {"min": -0.5, "max": 0.8, "palette": ["red", "white", "green"]}

Map2 = geemap.Map()
Map2.centerObject(AOI, 9)
Map2.addLayer(last_img.select('NBR'), nbr_vis, "NBR - latest year")
Map2

## 3. Year-over-year change detection

In [ ]:
change_pairs = build_change_series(annual_stack, band="NBR")
print([year for year, _ in change_pairs])

## 4. Spatial aggregation and net balance

Requires a spatial unit layer (`units_fc`) with a `unit_id` property — e.g. a hexagon grid or an MRC subdivision. Placeholder: generate a grid with `geemap` or upload your own asset.

In [ ]:
# Placeholder: replace with the real spatial unit layer
# units_fc = ee.FeatureCollection("projects/your-gee-project-id/assets/hex_grid")

results = []
for year, classified in change_pairs:
    # stats = aggregate_by_units(classified, units_fc, scale=10)
    # df_year = geemap.ee_to_df(stats)
    # df_year['year'] = year
    # results.append(df_year)
    pass

# balance_df = pd.concat(results, ignore_index=True)
# balance_df.to_csv('../figures/net_balance_by_unit.csv', index=False)

## 5. Result: cumulative net balance

Final chart for the LinkedIn post: annual loss vs recovery area, and cumulative net balance.

In [ ]:
import matplotlib.pyplot as plt

# balance_summary = balance_df.groupby('year')[['loss_ha', 'recovery_ha', 'net_balance_ha']].sum()
# balance_summary['net_balance_cumsum'] = balance_summary['net_balance_ha'].cumsum()

# fig, ax = plt.subplots(figsize=(9, 5))
# balance_summary[['loss_ha', 'recovery_ha']].plot(kind='bar', ax=ax)
# ax.set_ylabel('Hectares')
# ax.set_title('Annual loss vs recovery — managed forest of Quebec')
# plt.tight_layout()
# plt.savefig('../figures/annual_loss_recovery.png', dpi=200)
# plt.show()

## 6. Aboveground carbon level (GEDI L4B)

GEDI is a sparse lidar product (25 m footprints), not a dense annual time series. The L4B gridded product (1 km) provides mean aboveground biomass density (Mg/ha), converted here to carbon using the IPCC default fraction (0.47). Coverage gap: March 2023 - April 2024 (instrument stored on the ISS).

In [ ]:
from gedi_utils import get_carbon_density, aggregate_carbon_by_units, carbon_in_change_zones

carbon_density = get_carbon_density(AOI)

carbon_vis = {"min": 0, "max": 100, "palette": ["white", "yellow", "darkgreen"]}
Map3 = geemap.Map()
Map3.centerObject(AOI, 8)
Map3.addLayer(carbon_density, carbon_vis, "Aboveground carbon (Mg C/ha)")
Map3

## 7. Carbon by spatial unit and cross with change zones

In [ ]:
# carbon_by_unit = aggregate_carbon_by_units(carbon_density, units_fc, scale=1000)
# carbon_df = geemap.ee_to_df(carbon_by_unit)
# carbon_df.to_csv('../figures/carbon_by_unit.csv', index=False)

# Using the most recent available change classification:
# last_year, last_classified = change_pairs[-1]
# zones = carbon_in_change_zones(carbon_density, last_classified)

# carbon_loss_stats = zones['loss'].multiply(ee.Image.pixelArea().divide(10000)).reduceRegion(
#     reducer=ee.Reducer.sum(), geometry=AOI, scale=1000, maxPixels=1e13
# )
# print('Estimated carbon (Mg) in loss zones:', carbon_loss_stats.getInfo())

## 8. Combined result for LinkedIn

Two final visuals: (1) net loss/recovery balance map, (2) aboveground carbon map with loss zones highlighted — message: how much carbon is at risk in the highest-loss areas.